# Install libraries

In [ ]:
# Install required libraries
!pip install torch torchvision segmentation-models-pytorch kaggle --quiet

# Import necessary libraries
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from segmentation_models_pytorch import Unet
from segmentation_models_pytorch.encoders import get_preprocessing_fn
from PIL import Image
import numpy as np
import os
import zipfile

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 43.4 MB/s eta 0:00:00


# decide device


In [ ]:
# Set up device for training (GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


# Authenticate Kaggle


In [ ]:
# Authenticate Kaggle (Ensure you have your kaggle.json API key file)
!mkdir -p ~/.kaggle
!echo '{"username":"hafijulhoque987 ","key":"ba267fc402273b17f82059844f85fe32"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json


# Downloading the dataset

In [ ]:
# Download the dataset from Kaggle
!kaggle datasets download -d nikhilpandey360/chest-xray-masks-and-labels

# Extract the dataset
with zipfile.ZipFile('chest-xray-masks-and-labels.zip', 'r') as zip_ref:
    zip_ref.extractall('./chest_xray_data')

# Verify the dataset extraction
print("Dataset files extracted successfully!")


Dataset URL: https://www.kaggle.com/datasets/nikhilpandey360/chest-xray-masks-and-labels
License(s): CC0-1.0
100% 9.57G/9.58G [01:05<00:00, 196MB/s]
100% 9.58G/9.58G [01:05<00:00, 158MB/s]
Dataset files extracted successfully!


# Directory


In [ ]:
# Define paths to images and masks within the extracted directory
image_dir = './chest_xray_data/Lung Segmentation/CXR_png'
mask_dir = './chest_xray_data/Lung Segmentation/masks'

# Transformer

In [ ]:
import copy
import math
from typing import List, Optional
import torch
import torch.nn.functional as F
from torch import Tensor, nn

class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, x, mask=None):
        if mask is None:
            mask = torch.zeros((x.size(0), x.size(2), x.size(3)), device=x.device, dtype=torch.bool)
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack(
            (pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4
        ).flatten(3)
        pos_y = torch.stack(
            (pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4
        ).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos

class Transformer(nn.Module):
    def __init__(
        self,
        d_model=512,
        nhead=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
        return_intermediate_dec=False,
    ):
        super().__init__()

        encoder_layer = TransformerEncoderLayer(
            d_model, nhead, dim_feedforward, dropout, activation, normalize_before
        )
        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers, encoder_norm)

        decoder_layer = TransformerDecoderLayer(
            d_model, nhead, dim_feedforward, dropout, activation, normalize_before
        )
        decoder_norm = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(
            decoder_layer,
            num_decoder_layers,
            decoder_norm,
            return_intermediate=return_intermediate_dec,
        )

        self._reset_parameters()

        self.d_model = d_model
        self.nhead = nhead

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, mask, query_embed, query_pos, value_pos):
        # flatten NxCxHxW to HWxNxC
        bs, c, l = query_embed.shape
        query_embed = query_embed.permute(2, 0, 1)
        query_pos = query_pos.unsqueeze(0).expand(bs,-1,-1).permute(2, 0, 1)
        value=src.flatten(2).permute(2, 0, 1)
        value_pos=value_pos.flatten(2).permute(2, 0, 1)
        if mask is not None:
            mask = mask.flatten(1)

        #tgt = torch.zeros_like(query_embed)
        memory = self.encoder(value, src_key_padding_mask=mask, pos=value_pos)
        hs = self.decoder(
            tgt=query_embed, memory=memory, memory_key_padding_mask=mask, pos=value_pos, query_pos=query_pos
        )
        return hs.transpose(1, 2).transpose(0, 2)


class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = _get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm

    def forward(
        self,
        src,
        mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        output = src

        for layer in self.layers:
            output = layer(
                output, src_mask=mask, src_key_padding_mask=src_key_padding_mask, pos=pos
            )

        if self.norm is not None:
            output = self.norm(output)

        return output


class TransformerDecoder(nn.Module):
    def __init__(self, decoder_layer, num_layers, norm=None, return_intermediate=False):
        super().__init__()
        self.layers = _get_clones(decoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm
        self.return_intermediate = return_intermediate

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        output = tgt

        intermediate = []

        for layer in self.layers:
            output = layer(
                tgt=output,
                memory=memory,
                tgt_mask=tgt_mask,
                memory_mask=memory_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=memory_key_padding_mask,
                pos=pos,
                query_pos=query_pos,
            )
            if self.return_intermediate:
                intermediate.append(self.norm(output))

        if self.norm is not None:
            output = self.norm(output)
            if self.return_intermediate:
                intermediate.pop()
                intermediate.append(output)

        if self.return_intermediate:
            return torch.stack(intermediate)

        return output


class TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(
            q, k, value=src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

    def forward_pre(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        src2 = self.norm1(src)
        q = k = self.with_pos_embed(src2, pos)
        src2 = self.self_attn(
            q, k, value=src2, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src = src + self.dropout1(src2)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src2))))
        src = src + self.dropout2(src2)
        return src

    def forward(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        return self.forward_post(src, src_mask, src_key_padding_mask, pos)


class TransformerDecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        q = k = self.with_pos_embed(tgt, query_pos)
        tgt2 = self.self_attn(
            q, k, value=tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask
        )[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        tgt2 = self.multihead_attn(
            query=self.with_pos_embed(tgt, query_pos),
            key=self.with_pos_embed(memory, pos),
            value=memory,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
        )[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)

        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        return tgt

    def forward_pre(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        tgt2 = self.norm1(tgt)
        q = k = self.with_pos_embed(tgt2, query_pos)
        tgt2 = self.self_attn(
            q, k, value=tgt2, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask
        )[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt2 = self.norm2(tgt)
        tgt2 = self.multihead_attn(
            query=self.with_pos_embed(tgt2, query_pos),
            key=self.with_pos_embed(memory, pos),
            value=memory,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
        )[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt2 = self.norm3(tgt)
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt2))))
        tgt = tgt + self.dropout3(tgt2)
        return tgt

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        if self.normalize_before:
            return self.forward_pre(
                tgt,
                memory,
                tgt_mask,
                memory_mask,
                tgt_key_padding_mask,
                memory_key_padding_mask,
                pos,
                query_pos,
            )
        return self.forward_post(
            tgt,
            memory,
            tgt_mask,
            memory_mask,
            tgt_key_padding_mask,
            memory_key_padding_mask,
            pos,
            query_pos,
        )


def _get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])


def _get_activation_fn(activation):
    """Return an activation function given a string"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if activation == "glu":
        return F.glu
    raise RuntimeError(f"activation should be relu/gelu, not {activation}.")

# Unet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class MultiScaleAttention(nn.Module):
    def __init__(self, in_channels):
        super(MultiScaleAttention, self).__init__()
        self.scale1 = nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1)
        self.scale3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1)
        self.scale5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, stride=1, padding=2)

        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        scale1_out = self.scale1(x)
        scale3_out = self.scale3(x)
        scale5_out = self.scale5(x)

        combined = torch.stack([scale1_out, scale3_out, scale5_out], dim=1)  # Shape: (B, 3, C, H, W)
        attention_weights = self.softmax(combined.mean(dim=(3, 4), keepdim=True))  # Shape: (B, 3, C, 1, 1)

        output = (combined * attention_weights).sum(dim=1)  # Weighted sum of scales
        return output

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels, feature_dims):
        super(UNet, self).__init__()

        self.encoder1 = DoubleConv(in_channels, feature_dims)
        self.encoder2 = DoubleConv(feature_dims, feature_dims * 2)
        self.encoder3 = DoubleConv(feature_dims * 2, feature_dims * 4)
        self.encoder4 = DoubleConv(feature_dims * 4, feature_dims * 8)

        self.embed = nn.Embedding(feature_dims * 16, 1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.pe_layer = PositionEmbeddingSine(feature_dims * 16 // 2, normalize=True)
        self.transformer = Transformer(
            d_model=feature_dims * 16,
            dropout=0.1,
            nhead=4,
            dim_feedforward=feature_dims * 4,
            num_encoder_layers=0,
            num_decoder_layers=1,
            normalize_before=False,
            return_intermediate_dec=False,
        )

        self.bottleneck_conv = DoubleConv(feature_dims * 8, feature_dims * 16)

        self.attention4 = MultiScaleAttention(feature_dims * 8)
        self.attention3 = MultiScaleAttention(feature_dims * 4)
        self.attention2 = MultiScaleAttention(feature_dims * 2)
        self.attention1 = MultiScaleAttention(feature_dims)

        self.upconv4 = nn.ConvTranspose2d(feature_dims * 16, feature_dims * 8, kernel_size=2, stride=2)
        self.decoder4 = DoubleConv(feature_dims * 16, feature_dims * 8)
        self.upconv3 = nn.ConvTranspose2d(feature_dims * 8, feature_dims * 4, kernel_size=2, stride=2)
        self.decoder3 = DoubleConv(feature_dims * 8, feature_dims * 4)
        self.upconv2 = nn.ConvTranspose2d(feature_dims * 4, feature_dims * 2, kernel_size=2, stride=2)
        self.decoder2 = DoubleConv(feature_dims * 4, feature_dims * 2)
        self.upconv1 = nn.ConvTranspose2d(feature_dims * 2, feature_dims, kernel_size=2, stride=2)
        self.decoder1 = DoubleConv(feature_dims * 2, feature_dims)

        self.final_conv = nn.Conv2d(feature_dims, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool(enc1))
        enc3 = self.encoder3(self.pool(enc2))
        enc4 = self.encoder4(self.pool(enc3))

        # Bottleneck
        bottleneck = self.pool(enc4)
        bottleneck = self.bottleneck_conv(bottleneck)

        b, c, h, w = bottleneck.shape
        bottleneck_avg = bottleneck.mean(dim=(2, 3), keepdim=True).squeeze(-1)
        bottleneck_pe = self.pe_layer(bottleneck)

        transformer_output = self.transformer(bottleneck, None, bottleneck_avg, self.embed.weight, bottleneck_pe)
        transformer_output_expanded = transformer_output.unsqueeze(-1)
        bottleneck = bottleneck * transformer_output_expanded

        # Decoder with Multi-Scale Attention
        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, self.attention4(enc4)), dim=1)
        dec4 = self.decoder4(dec4)

        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, self.attention3(enc3)), dim=1)
        dec3 = self.decoder3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, self.attention2(enc2)), dim=1)
        dec2 = self.decoder2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, self.attention1(enc1)), dim=1)
        dec1 = self.decoder1(dec1)

        return torch.sigmoid(self.final_conv(dec1))


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torchvision.models import resnet50, resnet101
from torchvision.models._utils import IntermediateLayerGetter

class FrozenBatchNorm2d(torch.nn.Module):
    def __init__(self, n):
        super(FrozenBatchNorm2d, self).__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,
                              missing_keys, unexpected_keys, error_msgs):
        num_batches_tracked_key = prefix + 'num_batches_tracked'
        if num_batches_tracked_key in state_dict:
            del state_dict[num_batches_tracked_key]

        super(FrozenBatchNorm2d, self)._load_from_state_dict(
            state_dict, prefix, local_metadata, strict,
            missing_keys, unexpected_keys, error_msgs)

    def forward(self, x):
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        eps = 1e-5
        scale = w * (rv + eps).rsqrt()
        bias = b - rm * scale
        return x * scale + bias


class BackboneBase(nn.Module):
    def __init__(self, backbone: nn.Module, train_backbone: bool, num_channels: int, return_interm_layers: bool):
        super().__init__()
        for name, parameter in backbone.named_parameters():
            if not train_backbone or 'layer2' not in name and 'layer3' not in name and 'layer4' not in name:
                parameter.requires_grad_(False)
        if return_interm_layers:
            return_layers = {"layer1": "0", "layer2": "1", "layer3": "2", "layer4": "3"}
        else:
            return_layers = {'layer4': "0"}
        self.body = IntermediateLayerGetter(backbone, return_layers=return_layers)
        self.num_channels = num_channels

    def forward(self, x):
        x = self.body(x)
        return x

resnets_dict = {
    'resnet50': resnet50,
    'resnet101': resnet101,
}

class Backbone(BackboneBase):
    def __init__(self, name: str, train_backbone: bool, return_interm_layers: bool, dilation: list):
        backbone = resnets_dict[name](
            pretrained=True, replace_stride_with_dilation=dilation, norm_layer=FrozenBatchNorm2d
        )
        num_channels = 512 if name in ('resnet18', 'resnet34') else 2048
        super().__init__(backbone, train_backbone, num_channels, return_interm_layers)

In [ ]:
import torch
import torch.nn as nn
from .backbone_utils import Backbone
from .transformer import PositionEmbeddingSine
from .transformer import Transformer

class MSR(nn.Module):
    def __init__(self, layers, num_classes=2, reduce_dim=256):
        super(MSR, self).__init__()
        self.backbone = Backbone(
            'resnet{}'.format(layers),
            train_backbone=False,
            return_interm_layers=True,
            dilation=[False, True, True]
        )
        self.embed_cat=nn.Embedding(reduce_dim,1)
        self.embed_3=nn.Embedding(reduce_dim,1)
        self.pe_layer_cat=PositionEmbeddingSine(reduce_dim//2, normalize=True)
        self.pe_layer_3=PositionEmbeddingSine(reduce_dim//2, normalize=True)

        self.transformer_cat = Transformer(
            d_model=reduce_dim,
            dropout=0.1,
            nhead=4,
            dim_feedforward=reduce_dim//4,
            num_encoder_layers=0,
            num_decoder_layers=1,
            normalize_before=False,
            return_intermediate_dec=False,
        )

        self.transformer_3 = Transformer(
            d_model=reduce_dim,
            dropout=0.1,
            nhead=4,
            dim_feedforward=reduce_dim//4,
            num_encoder_layers=0,
            num_decoder_layers=1,
            normalize_before=False,
            return_intermediate_dec=False,
        )
        self.conv_red_1 = nn.Sequential(
            nn.Conv2d(512, reduce_dim//2, kernel_size=1, padding=0, bias=False)
        )
        self.conv_red_2 = nn.Sequential(
            nn.Conv2d(1024, reduce_dim//2, kernel_size=1, padding=0, bias=False)
        )
        self.conv_red_3 = nn.Sequential(
            nn.Conv2d(2048, reduce_dim, kernel_size=1, padding=0, bias=False)
        )


        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(reduce_dim*2, reduce_dim)  # Assuming last feature map size
        self.fc_2 = nn.Linear(reduce_dim, num_classes)  # Assuming last feature map size
        self.num_classes = num_classes

    def forward(self, x):
        # Backbone feature extraction
        back_x = self.backbone(x)
        red_back_1 = self.conv_red_1(back_x['1'])
        red_back_2 = self.conv_red_2(back_x['2'])

        red_back_cat = torch.cat((red_back_1, red_back_2), dim=1)
        avg_back_cat = self.avgpool(red_back_cat)

        red_back_3 = self.conv_red_3(back_x['3'])
        avg_back_3 = self.avgpool(red_back_3)

        masking=None
        query_pos = self.embed_cat.weight

        key_embed = red_back_cat
        query_embed = avg_back_cat.squeeze(-1)
        key_pos = self.pe_layer_cat(red_back_cat)

        fg_embed_cat=self.transformer_cat(key_embed,masking,query_embed,query_pos,key_pos)

        query_pos = self.embed_3.weight
        key_embed = red_back_3
        query_embed = avg_back_3.squeeze(-1)
        key_pos = self.pe_layer_3(red_back_3)
        fg_embed_3=self.transformer_3(key_embed,masking,query_embed,query_pos,key_pos)


        #out_back_cat = (torch.einsum("bchw,bcl->blhw",red_back_cat,fg_embed_cat)).permute(0, 2, 3, 1)
        #out_back_3 = (torch.einsum("bchw,bcl->blhw",red_back_3,fg_embed_3)).permute(0, 2, 3, 1)



        # Concatenate along the last dimension
        out = torch.cat((fg_embed_cat, fg_embed_3), dim=1)

        out_1 = torch.flatten(out, 1)  # Flatten the feature maps

        out_1 = self.fc(out_1)  # Fully connected layer
        out_1 = self.fc_2(out_1)  # Fully connected layer
        return out_1

    def get_backbone_params(self):
        return self.backbone.parameters()

    def get_fc_params(self):
        return self.fc.parameters()

In [ ]:
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image


class SegmentationDataset(Dataset):


    def __init__(self, image_dir, mask_dir, transform=None):
        """
        Args:
            image_dir (str): Path to the directory containing images.
            mask_dir (str): Path to the directory containing masks.
            transform (callable, optional): Transformation to apply to both images and masks.
        """
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

        # Include only images with corresponding masks
        self.images = [
            img_name for img_name in os.listdir(image_dir)
            if os.path.exists(os.path.join(mask_dir, img_name))
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Load image and corresponding mask
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        image = Image.open(img_path).convert("L")  # Ensure grayscale
        mask = Image.open(mask_path).convert("L")  # Ensure binary

        # Apply transformations
        if self.transform is not None:
            image, mask = self.transform(image, mask)

        return image, mask


class JointTransform:


    def __init__(self, image_transform=None, mask_transform=None):
        """
        Args:
            image_transform (callable, optional): Transformation to apply to images.
            mask_transform (callable, optional): Transformation to apply to masks.
        """
        self.image_transform = image_transform
        self.mask_transform = mask_transform if mask_transform else image_transform

    def __call__(self, image, mask):
        """
        Apply transformations with the same random seed for image and mask.

        Args:
            image (PIL.Image): Input image.
            mask (PIL.Image): Corresponding mask.

        Returns:
            Transformed image and mask.
        """
        seed = random.randint(0, 2**32)
        random.seed(seed)
        torch.manual_seed(seed)
        image = self.image_transform(image)

        random.seed(seed)
        torch.manual_seed(seed)
        mask = self.mask_transform(mask)

        return image, mask


def get_data_loader(base_dir, batch_size, img_size=(512, 512), val_split=0.2):
    """
    Function to prepare DataLoader objects for training, validation, and testing datasets.

    Args:
        base_dir (str): Base directory containing train/test subdirectories.
        batch_size (int): Batch size for DataLoader.
        img_size (tuple): Size to resize images and masks.
        val_split (float): Fraction of the training dataset to use for validation.

    Returns:
        train_loader: DataLoader for the training set.
        val_loader: DataLoader for the validation set.
        test_loader: DataLoader for the testing set.
    """
    # Directories
    train_image_dir = os.path.join(base_dir, "CXR_png")
    train_mask_dir = os.path.join(base_dir, "masks")
    test_image_dir = os.path.join(base_dir, "test")
    test_mask_dir = os.path.join(base_dir, "test")

    base_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

    mask_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
])


    # Training and testing transformations
    train_transform = JointTransform(image_transform=base_transform, mask_transform=mask_transform)
    test_transform = JointTransform(
        image_transform=transforms.Compose([transforms.Resize(img_size), transforms.ToTensor()]),
        mask_transform=transforms.Compose([transforms.Resize(img_size), transforms.ToTensor()])
    )

    # Full training dataset
    full_train_dataset = SegmentationDataset(
        image_dir=train_image_dir,
        mask_dir=train_mask_dir,
        transform=train_transform
    )

    # Split into training and validation datasets
    train_size = int((1 - val_split) * len(full_train_dataset))
    val_size = len(full_train_dataset) - train_size
    train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

    # Testing dataset
    test_dataset = SegmentationDataset(
        image_dir=test_image_dir,
        mask_dir=test_mask_dir,
        transform=test_transform
    )

    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader



## Combined loss function


In [ ]:
def combined_loss(outputs, targets, dice_weight=0.5, bce_weight=0.5):

    # Binary Cross-Entropy Loss
    bce_loss = nn.BCEWithLogitsLoss()(outputs, targets)

    # Dice Loss
    smooth = 1e-5  # To prevent division by zero
    intersection = (outputs * targets).sum(dim=(2, 3))
    union = outputs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice_loss = 1 - (2.0 * intersection + smooth) / (union + smooth)
    dice_loss = dice_loss.mean()  # Mean over batch

    # Combine BCE and Dice Loss
    return dice_weight * dice_loss + bce_weight * bce_loss


## Dice loss function


In [ ]:
def dice_coefficient(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    target = target.float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    if union.item() == 0:
        return 1.0
    return 2.0 * intersection / union

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim



def train(model, train_loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_dice = 0.0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        print(f"Input shape: {inputs.shape}")  # Debugging: Check input shape
        inputs, targets = inputs.to(device), targets.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Compute loss
        loss = criterion(outputs, targets)
        loss.backward()

        # Optimize the weights
        optimizer.step()

        # Update metrics
        running_loss += loss.item()
        dice = dice_coefficient(outputs, targets)
        running_dice += dice.item()

    return running_loss / len(train_loader), running_dice / len(train_loader)


def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_dice = 0.0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)

            loss = criterion(outputs, targets)
            running_loss += loss.item()

            dice = dice_coefficient(outputs, targets)
            running_dice += dice.item()

    return running_loss / len(val_loader), running_dice / len(val_loader)






## Training loop


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels=1, out_channels=1, feature_dims=64).to(device)  # Using grayscale images

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Load Data Loaders
base_dir = "./chest_xray_data/Lung Segmentation"  # Path to your dataset directory
batch_size = 8
train_loader, val_loader, test_loader = get_data_loader(base_dir, batch_size, img_size=(512, 512))

# Training Loop
epochs = 15
best_val_loss = float('inf')


for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    start_time = time.time()

    # Training Phase
    train_loss, train_dice = train(model, train_loader, optimizer, criterion, device)

    # Validation Phase
    val_loss, val_dice = validate(model=model, val_loader=val_loader, criterion=criterion, device=device)

    # Save the Best Model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        os.makedirs("checkpoints", exist_ok=True)
        torch.save(model.state_dict(), "checkpoints/best_model.pth")
        print(f"Saved best model with validation loss: {best_val_loss:.4f}")

    end_time = time.time()
    print(f"Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")
    print(f"Epoch Time: {end_time - start_time:.2f} seconds\n")



# Generating masks

In [ ]:
import os
from PIL import Image
import numpy as np

def generate_masks(model, test_loader, device, output_dir, threshold=0.5):
    model.eval()

    os.makedirs(output_dir, exist_ok=True)

    model.to(device)

    with torch.no_grad():
        for i, (inputs, targets) in enumerate(test_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            predictions = (outputs > threshold).float()

            for j in range(inputs.size(0)):
                pred_mask = predictions[j].cpu().numpy().squeeze()
                image_name = test_loader.dataset.images[i * test_loader.batch_size + j]
                save_path = os.path.join(output_dir, image_name.replace(".jpg", "_mask.png"))

                # Save the predicted mask
                pred_mask = (pred_mask * 255).astype(np.uint8)  # Scale to 0-255
                Image.fromarray(pred_mask).save(save_path)

                print(f'Saved mask for {image_name} to {save_path}')


generate_masks(model,test_loader,device,"generated_masks")


In [ ]:
import matplotlib.pyplot as plt

# Visualization Function
def visualize_predictions(model, val_loader, device, n=5):
    model.eval()  # Set model to evaluation mode
    with torch.no_grad():
        inputs, targets = next(iter(val_loader))  # Get a batch of validation data
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)  # Apply sigmoid for binary mask prediction
        outputs = (outputs > 0.5).float()  # Thresholding to create binary masks

        # Plot the first n samples
        for i in range(min(n, inputs.size(0))):
            plt.figure(figsize=(15, 5))

            # Input Image
            plt.subplot(1, 3, 1)
            plt.title("Input Image")
            if inputs[i].shape[0] == 1:  # If grayscale image
                plt.imshow(inputs[i].cpu().squeeze(), cmap='gray')
            else:  # If RGB image
                plt.imshow(inputs[i].cpu().permute(1, 2, 0))  # Convert CHW to HWC

            # Ground Truth Mask
            plt.subplot(1, 3, 2)
            plt.title("Ground Truth Mask")
            plt.imshow(targets[i].cpu().squeeze(), cmap='gray')

            # Predicted Mask
            plt.subplot(1, 3, 3)
            plt.title("Predicted Mask")
            plt.imshow(outputs[i].cpu().squeeze(), cmap='gray')

            plt.show()

# Call the function
visualize_predictions(model, test_loader, device, n=5)
